In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf

from tensorflow.keras.layers import (Input, Dense, Concatenate,
                                     GlobalAveragePooling2D, Conv2D,
                                     MaxPooling2D, Dropout)
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    EfficientNetB3, MobileNetV2, ResNet50, VGG19, DenseNet121
)
from sklearn.model_selection import train_test_split

# ================================
# CONFIG
# ================================
IMG_SIZE = 224
NUM_CLASSES = 5
DATA_DIR = "data/"

# ================================
# DATASET LOADER
# ================================
def load_dataset(data_dir):

    if not os.path.exists(data_dir):
        raise ValueError("Dataset folder 'data/' not found. Create it with class subfolders.")

    images = []
    labels = []

    class_names = sorted(os.listdir(data_dir))

    if len(class_names) == 0:
        raise ValueError("No class folders found inside 'data/'.")

    for label, class_name in enumerate(class_names):
        class_path = os.path.join(data_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                img = cv2.imread(img_path)

                if img is None:
                    continue

                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(label)

            except:
                continue

    if len(images) == 0:
        raise ValueError("No valid images found in dataset.")

    return images, labels


# ================================
# PREPROCESSING
# ================================
def preprocess_image(img):
    return img.astype("float32") / 255.0


# ================================
# SQUEEZENET
# ================================
def fire_module(x, squeeze, expand):

    squeeze_layer = Conv2D(squeeze, (1,1), activation='relu')(x)

    expand1 = Conv2D(expand, (1,1), activation='relu')(squeeze_layer)
    expand3 = Conv2D(expand, (3,3), padding='same', activation='relu')(squeeze_layer)

    return Concatenate()([expand1, expand3])


def SqueezeNet(input_shape=(224,224,3)):

    inputs = Input(shape=input_shape)

    x = Conv2D(96, (7,7), strides=2, padding='same', activation='relu')(inputs)
    x = MaxPooling2D((3,3), strides=2)(x)

    x = fire_module(x, 16, 64)
    x = fire_module(x, 16, 64)
    x = MaxPooling2D((3,3), strides=2)(x)

    x = fire_module(x, 32, 128)
    x = fire_module(x, 32, 128)
    x = MaxPooling2D((3,3), strides=2)(x)

    x = fire_module(x, 48, 192)
    x = fire_module(x, 48, 192)

    x = GlobalAveragePooling2D()(x)

    return Model(inputs, x, name="SqueezeNet")


# ================================
# LOAD BASE MODELS
# ================================
def get_models(input_shape):

    models = [
        EfficientNetB3(include_top=False, weights='imagenet', input_shape=input_shape),
        MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape),
        ResNet50(include_top=False, weights='imagenet', input_shape=input_shape),
        VGG19(include_top=False, weights='imagenet', input_shape=input_shape),
        DenseNet121(include_top=False, weights='imagenet', input_shape=input_shape),
        SqueezeNet(input_shape)
    ]

    for m in models:
        m.trainable = False

    return models


# ================================
# DFEN MODEL
# ================================
def DFEN(input_shape=(224,224,3)):

    inputs = Input(shape=input_shape)
    base_models = get_models(input_shape)

    features = []

    for model in base_models:
        x = model(inputs)

        if len(x.shape) == 4:
            x = GlobalAveragePooling2D()(x)

        features.append(x)

    fused = Concatenate()(features)

    x = Dense(256, activation='relu')(fused)
    x = Dropout(0.5)(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)

    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inputs, outputs)


# ================================
# MAIN
# ================================
if __name__ == "__main__":

    print("Loading dataset...")
    images, labels = load_dataset(DATA_DIR)

    print("Preprocessing...")
    X = np.array([preprocess_image(img) for img in images])
    y = np.array(labels)

    print("Splitting dataset...")
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print("Building model...")
    model = DFEN()

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    print("Training...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=8
    )

    print("Saving model...")
    model.save("DFEN_DR_model.h5")

    print("Training completed successfully.")

Loading dataset...


ValueError: Dataset folder 'data/' not found. Create it with class subfolders.